### Write programs to demonstrate the workings of the decision tree (DT) using ID3, C4.5, and CART algorithm. Use at least three different datasets and break them into 70% to train the DT models, and use the remaining 30% of the data to measure the performance of the models based on accuracy, precision, recall, and F1-score. Show a comparison table and discuss the performance outcomes

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X_diab = pd.read_csv("datasets/diabetes.csv")
X_gast = pd.read_csv("datasets/Gastric.csv")
X_iris = pd.read_csv("datasets/Iris.csv")
y_diab = X_diab.iloc[:,-1:]
y_gast = X_gast.iloc[:,-1:]
y_iris = X_iris.iloc[:,-1:]
X_diab.drop
y_diab

,Outcome
0,1
1,0
2,1
3,0
4,1
...,...
763,0
764,0
765,0
766,1


In [2]:
class DecisionTree:
    def __init__(self, alg_type='CART', max_depth=5):
        self.alg_type = alg_type
        self.max_depth = max_depth
        self.tree = None

    def _entropy(self, y):
        proportions = y.value_counts(normalize=True)
        return -sum(p * np.log2(p) for p in proportions if p > 0)

    def _gini(self, y):
        proportions = y.value_counts(normalize=True)
        return 1 - sum(p**2 for p in proportions)

    def _information_gain(self, parent_y, subsets_y):
        parent_entropy = self._entropy(parent_y)
        total_size = len(parent_y)
        weighted_entropy = sum((len(sub) / total_size) * self._entropy(sub) for sub in subsets_y)
        return parent_entropy - weighted_entropy

    def _gain_ratio(self, parent_y, subsets_y):
        gain = self._information_gain(parent_y, subsets_y)
        total_size = len(parent_y)
        # Split Information (Intrinsic Value)
        split_info = -sum((len(sub) / total_size) * np.log2(len(sub) / total_size) for sub in subsets_y if len(sub) > 0)
        return gain / split_info if split_info != 0 else 0

    def _best_split(self, X, y):
        best_gain = -1
        split_params = None
        
        for feature in X.columns:
            thresholds = X[feature].unique()
            for threshold in thresholds:
                left_idx = X[feature] <= threshold
                right_idx = ~left_idx
                
                if len(y[left_idx]) == 0 or len(y[right_idx]) == 0:
                    continue
                
                subsets = [y[left_idx], y[right_idx]]
                
                if self.alg_type == 'ID3':
                    gain = self._information_gain(y, subsets)
                elif self.alg_type == 'C4.5':
                    gain = self._gain_ratio(y, subsets)
                elif self.alg_type == 'CART':
                    # For CART, we minimize Gini (maximize reduction)
                    gain = self._gini(y) - (len(y[left_idx])/len(y) * self._gini(y[left_idx]) + 
                                           len(y[right_idx])/len(y) * self._gini(y[right_idx]))
                
                if gain > best_gain:
                    best_gain = gain
                    split_params = (feature, threshold)
                    
        return split_params

    def build_tree(self, X, y, depth=0):
        # Base cases: pure node, no features, or max depth
        if len(y.unique()) == 1 or depth >= self.max_depth or len(X.columns) == 0:
            return y.value_counts().idxmax()

        split = self._best_split(X, y)
        if not split:
            return y.value_counts().idxmax()

        feature, threshold = split
        left_idx = X[feature] <= threshold
        right_idx = ~left_idx

        node = {
            'feature': feature,
            'threshold': threshold,
            'left': self.build_tree(X[left_idx], y[left_idx], depth + 1),
            'right': self.build_tree(X[right_idx], y[right_idx], depth + 1)
        }
        return node

    def fit(self, X, y):
        self.tree = self.build_tree(X, y)

    def _predict_one(self, sample, tree):
        if not isinstance(tree, dict):
            return tree
        if sample[tree['feature']] <= tree['threshold']:
            return self._predict_one(sample, tree['left'])
        else:
            return self._predict_one(sample, tree['right'])

    def predict(self, X):
        return X.apply(lambda row: self._predict_one(row, self.tree), axis=1)

In [ ]:
# Function to evaluate all models on a dataset
def run_test(X, y, name):
    X = pd.DataFrame(data, columns=data.feature_names)
    y = pd.Series(data.target)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    results = []
    for alg in ['ID3', 'C4.5', 'CART']:
        model = DecisionTree(alg_type=alg, max_depth=4)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Performance Metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='macro')
        rec = recall_score(y_test, y_pred, average='macro')
        f1 = f1_score(y_test, y_pred, average='macro')
        
        results.append({'Dataset': name, 'Algorithm': alg, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1})
    
    return results

all_results = []
all_results.extend(run_test(df_diab, "Diabetes"))
all_results.extend(run_test(df_gast, "Gastric"))
all_results.extend(run_test(df_iris, "Iris"))

# Comparison Table
df_results = pd.DataFrame(all_results)
print(df_results.to_string(index=False))

AttributeError: 'DataFrame' object has no attribute 'data'